# PROJECT 1: GAUSS ELIMINATION AND BACK SUBSTITUTION

## I. STUDENT INFORMATION
* **Full Name:** Nguyễn Nhựt Huy
* **Student ID (MSSV):** 24127398
* **Class:** 24C06
* **Course:** Applied Mathematics and Statistics

## II. ALGORITHM OVERVIEW

---

### 1. Gauss Elimination Theory
* **Objective:** To systematically reduce an arbitrary augmented matrix $[A | b]$ into an upper triangular or Row Echelon Form (REF).
* **Mathematical Operations:** The method relies on three elementary row operations defined in Linear Algebra:
  1. Swapping the positions of two equations (rows).
  2. Multiplying an equation by a non-zero scalar.
  3. Adding a scalar multiple of one equation to another.
* **Partial Pivoting Mechanism:** In numerical computations, selecting the largest absolute coefficient in the operational column as the "pivot" is necessary to avoid dividing by zero or extremely small floating-point configurations, ensuring numerical stability.

---

### 2. Back Substitution Theory
* **Objective:** To extract the exact solution vector $\mathbf{x}$ once the augmented system is in Row Echelon Form.
* **System Classification (Kronecker-Capelli Theorem):** 
  * **Inconsistent (No Solution):** Occurs if a row simplifies to $0 = c$ (where $c \neq 0$). The rank of the coefficient matrix is strictly less than the rank of the augmented matrix.
  * **Consistent with a Unique Solution:** Occurs when the number of leading non-zero pivots equals the total number of variables. Each variable maps directly to a deterministic numerical scalar.
  * **Consistent with Infinitely Many Solutions:** Occurs when the number of pivots is less than the total number of variables. Columns without leading pivots are categorized as **Free Variables**, acting as independent algebraic parameters used to express the dependent pivot variables.

## III. Implementation

### Import necessary library

In [30]:
import numpy as np
import sympy as sp
import scipy.linalg as la
import random

### Support Functions

In [31]:
def swap_rows(A, i, j):
    temp = A[i].copy()
    A[i] = A[j]
    A[j] = temp

In [32]:
def multiply_row(A, i, k):
    A[i] = A[i] * k

In [33]:
def add_row_multiple(A, dest_row, src_row, k):
    A[dest_row] = A[dest_row] - k*A[src_row]

In [34]:
def is_zero(val, tol=1e-9):
    return abs(val) < tol

### Gauss elimination

In [35]:
def Gauss_elimination(A):
    cur_row = 0
    n, m = A.shape
    for j in range (0, m-1):
        if cur_row >= n:
            break

        max_val = 0 
        pivot_row = cur_row
        for i in range (cur_row, n):
            if abs(A[i, j]) > max_val:
                max_val = abs(A[i, j])
                pivot_row = i
        
        if is_zero(max_val):
            continue
        
        swap_rows(A, cur_row, pivot_row)
        
        multiply_row(A, cur_row, 1/A[cur_row, j])

        for k in range (cur_row+1, n):
            val = float(A[k ,j])
            add_row_multiple(A, k, cur_row, val)

        cur_row += 1

    return A

### Back substitution

In [36]:
def back_substitution(A):
    n, m = A.shape
    num_variables = m - 1

    # kiem tra pt vo nghiem
    for i in range (n-1, -1, -1):
        all_variables_zero = np.all([is_zero(A[i, j]) for j in range (num_variables)])
        left_zero = not is_zero(A[i, m-1])

        if (all_variables_zero and left_zero):
            return None
    
    # giai nghiem
    x_sym = [sp.symbols(f'x_{i+1}') for i in range(num_variables)]
    
    pivot_columns = []
    for i in range(n):
        for j in range(num_variables):
            if not is_zero(A[i, j]):
                pivot_columns.append(j)
                break

    num_pivots = len(pivot_columns)
    for i in range(num_pivots - 1, -1, -1):
        j = pivot_columns[i]

        left_sum = sum([A[i, k] * x_sym[k] for k in range(j+1, num_variables)])

        x_sym[j] = sp.simplify(A[i, m-1] - left_sum)


    return x_sym

## IV. Test cases and Comparisons

### Test cases

In [37]:
test_cases = [
    # Bai 1
    np.array([[1, 2, -1, -1],
              [2, 2, 1, 1],
              [3, 5, -2, -1]], dtype=float),
    
    # Bai 2
    np.array([[1, -2, -1, 1],
              [2, -3, 1, 6],
              [3, -5, 0, 7],
              [1, 0, 5, 9]], dtype=float),
    
    # Bai 3
    np.array([[1, 2, 0, 2, 6],
              [3, 5, -1, 6, 17],
              [2, 4, 1, 2, 12],
              [2, 0, -7, 11, 7]], dtype=float),
    
    # Bai 4
    np.array([[2, -4, -1, 1],
              [1, -3, 1, 1],
              [3, -5, -3, 2]], dtype=float),
    
    # Bai 5
    np.array([[1, 2, -2, 3],
              [3, -1, 1, 1],
              [-1, 5, -5, 5]], dtype=float),
    
    # Bai 6
    np.array([[2, -4, 6, 8],
              [1, -1, 1, -1],
              [1, -3, 4, 0]], dtype=float),
    
    # Bai 7
    np.array([[4, -2, -4, 2, 1],
              [6, -3, 0, -5, 3],
              [8, -4, 28, -44, 11],
              [-8, 4, -4, 12, -5]], dtype=float),
    
    # Bai 8
    np.array([[1, -2, 3, -3],
              [2, 2, 0, 0],
              [0, -3, 4, 1],
              [1, 0, 1, -1]], dtype=float),
    
    # Bai 9
    np.array([[3, -3, 3, -3],
              [-1, -5, 2, 4],
              [0, -4, 2, 2],
              [3, -1, 2, -4]], dtype=float),
    
    # Bai 10
    np.array([[1, -1, 1, -3, 0],
              [2, -1, 4, -2, 0]], dtype=float),
    
    # Bai 11
    np.array([[2, -3, 4, -1, 0],
              [6, 1, -8, 9, 0],
              [2, 6, 1, -1, 0]], dtype=float),
    
    # Bai 12
    np.array([[1, 6, 4, 0],
              [2, 4, -1, 0],
              [-1, 2, 5, 0]], dtype=float)
]

In [38]:
for idx, matrix in enumerate(test_cases, start=1):
    print(f"========================================")
    print(f"TEST CASE {idx}:")
    print(f"========================================")
    print("Initial Augmented Matrix:\n", matrix)
    print("-" * 40)
    
    A_lib = matrix[:, :-1]
    b_lib = matrix[:, -1]
    
    # Work on a copy for your custom implementation
    A_input = matrix.copy()
    echelon_matrix = Gauss_elimination(A_input)
    solution = back_substitution(echelon_matrix)
    
    print("Row Echelon Form Matrix:\n", echelon_matrix)
    print("-" * 40)
    
    print("--- [My Algorithm Result] ---")
    my_status = ""
    if solution is None:
        my_status = "No Solution"
        print("Status: Inconsistent System (No Solution)")
    else:
        all_symbols = set().union(*(expr.free_symbols for expr in solution if hasattr(expr, 'free_symbols')))
        if len(all_symbols) > 0:
            my_status = "Infinitely Many Solutions"
            print("Status: Underdetermined System (Infinitely Many Solutions)")
            print(f"Free Variables: {', '.join(str(sym) for sym in sorted(all_symbols, key=lambda s: s.name))}")
            print("Solution expressions:")
            for i, expr in enumerate(solution):
                print(f"   x_{i+1} = {sp.simplify(expr)}")
        else:
            my_status = "Unique Solution"
            print("Status: Consistent System (Unique Solution)")
            print("Solution values:")
            for i, val in enumerate(solution):
                float_val = float(val)
                formatted_val = int(float_val) if float_val.is_integer() else round(float_val, 4)
                print(f"   x_{i+1} = {formatted_val}")
                
    print("-" * 40)
    

    print("\n--- [SymPy Library Result] ---")

    m, n_plus_1 = matrix.shape
    n = n_plus_1 - 1

    variables = sp.symbols(f'x1:{n+1}')

    A_sym = sp.Matrix(matrix[:, :-1])
    b_sym = sp.Matrix(matrix[:, -1])

    library_solution = sp.linsolve((A_sym, b_sym), variables)

    if len(library_solution) == 0:
        lib_status = "No Solution"
        print("Status: Inconsistent System (No Solution)")

    else:
        sol_tuple = next(iter(library_solution))

        free_symbols = set()
        for expr in sol_tuple:
            free_symbols |= expr.free_symbols

        if free_symbols:
            lib_status = "Infinitely Many Solutions"
            print("Status: Underdetermined System (Infinitely Many Solutions)")
            print("Free Variables:",
                ", ".join(str(s) for s in sorted(free_symbols,
                                                key=lambda x: x.name)))

            for i, expr in enumerate(sol_tuple):
                print(f"   x_{i+1} = {sp.simplify(expr)}")

        else:
            lib_status = "Unique Solution"
            print("Status: Consistent System (Unique Solution)")

            for i, val in enumerate(sol_tuple):
                val = float(val)
                val = int(val) if val.is_integer() else round(val, 4)
                print(f"   x_{i+1} = {val}")

TEST CASE 1:
Initial Augmented Matrix:
 [[ 1.  2. -1. -1.]
 [ 2.  2.  1.  1.]
 [ 3.  5. -2. -1.]]
----------------------------------------
Row Echelon Form Matrix:
 [[ 1.          1.66666667 -0.66666667 -0.33333333]
 [-0.          1.         -1.75       -1.25      ]
 [ 0.          0.          1.         -1.        ]]
----------------------------------------
--- [My Algorithm Result] ---
Status: Consistent System (Unique Solution)
Solution values:
   x_1 = 4.0
   x_2 = -3.0
   x_3 = -1.0
----------------------------------------

--- [SymPy Library Result] ---
Status: Consistent System (Unique Solution)
   x_1 = 4.0
   x_2 = -3.0
   x_3 = -1.0
TEST CASE 2:
Initial Augmented Matrix:
 [[ 1. -2. -1.  1.]
 [ 2. -3.  1.  6.]
 [ 3. -5.  0.  7.]
 [ 1.  0.  5.  9.]]
----------------------------------------
Row Echelon Form Matrix:
 [[ 1.00000000e+00 -1.66666667e+00  0.00000000e+00  2.33333333e+00]
 [ 0.00000000e+00  1.00000000e+00  3.00000000e+00  4.00000000e+00]
 [ 0.00000000e+00  0.00000000e+0

## VI. IMPLEMENTATION IDEAS AND FUNCTION DESCRIPTIONS

---

### 1. Code Architecture & Implementation Strategy

#### A. Data Models and Precision Control
* **Vectorized Processing:** The system structures inputs as `numpy.ndarray` objects containing a native `float` data type. This architecture enables efficient computational column slicing and element-wise array mutations.
* **Epsilon Thresholding:** To prevent floating-point accumulation rounding faults (e.g., computer artifacts rendering a theoretical $0$ as $1 \times 10^{-16}$), raw conditional gates are replaced with an explicit tolerance function thresholding values within a boundary layer of $1 \times 10^{-9}$.

#### B. Program Execution Workflow
1. **The Core Loop:** The forward elimination engine operates incrementally down the matrix boundaries via an active index tracker (`cur_row`).
2. **In-Place Restructuring:** Instead of returning newly allocated matrices during operations, row mutations are computed **in-place** directly through memory-efficient utility subroutines (`swap_rows`, `multiply_row`, `add_row_multiple`) to handle large system benchmarks.
3. **Symbolic Fallback Integration:** To handle infinite variable dimensions cleanly without losing algebraic precision, the backward recovery engine initializes an array of `sympy` symbolic variables. If a variable column lacks a pivot point, its symbol remains untouched, dynamically generating parametric solution formulas.

---

### 2. Detailed Function Descriptions

Below is the technical documentation mapping out the exact operational parameters, arguments, and expected behaviors for every routine written in the source code.

#### A. Helper Subroutines

| Function Name & Signature | Programmatic Utility | Input Parameters | Return Value |
| :--- | :--- | :--- | :--- |
| `swap_rows(A, i, j)` | Exchanges row `i` with row `j` directly inside the numpy matrix reference. | <ul><li>`A` (np.array): Target matrix</li><li>`i` (int): First row index</li><li>`j` (int): Second row index</li></ul> | `None` (Mutates object in place) |
| `multiply_row(A, i, k)` | Scales row `i` by multiplying all its internal values by scalar factor $k$. | <ul><li>`A` (np.array): Target matrix</li><li>`i` (int): Active row index</li><li>`k` (float): Scalar factor</li></ul> | `None` (Mutates object in place) |
| `add_row_multiple(A, dest_row, src_row, k)` | Performs row elimination operation: $A[\text{dest}] = A[\text{dest}] - k \times A[\text{src}]$. | <ul><li>`A` (np.array): Target matrix</li><li>`dest_row` (int): Target row index</li><li>`src_row` (int): Source reference row</li><li>`k` (float): Scalar multiplier</li></ul> | `None` (Mutates object in place) |
| `is_zero(val, tol=1e-9)` | Tests numerical proximity to zero to determine if a value represents a structural mathematical zero. | <ul><li>`val` (float): Number to evaluate</li><li>`tol` (float): Evaluation margin (Default: `1e-9`)</ul> | `bool`: `True` if $\|val\| < tol$, else `False`. |

#### B. Core Solving Engines

### `Gauss_elimination(A)`
* **Functional Description:** Coordinates partial row pivoting actions and iterative row subtraction loops to step-reduce any arbitrary rectangular matrix into its corresponding row echelon matrix format.
* **Input:** `A` (`numpy.ndarray`): The initial $n \times m$ extended matrix $[A\|b]$.
* **Output:** `numpy.ndarray`: The in-place modified matrix transformed into standard Row Echelon Form.

### `back_substitution(A)`
* **Functional Description:** Triggers the backward substitution pass over a reduced row echelon array. It scans for system inconsistency signatures, extracts leading pivot locations, and returns mathematical value representations.
* **Input:** `A` (`numpy.ndarray`): A verified $n \times m$ Row Echelon Form matrix.
* **Output:** * `None`: If an equation simplifies to an impossible configuration ($0 = c$), returning an explicit warning to standard console logs.
  * `list`: A sequence containing numerical values (for unique solutions) or algebraic SymPy formulas expressing pivot variables via free parameters (for infinite solution sets).